In [1]:
# conda activate genomic_tools

import os
import glob
import json
import pickle
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', None)

## Prep data from UCSC genome browser tracks

In [2]:
track_df_list = []
keep = ["unipLocTransMemb", "unipOther", "unipLocSignal", "unipModif", "unipRepeat", 
        "unipLocCytopl", "unipChain", "unipLocExtra", "unipDomain", "unipStruct", 
        "unipDisulfBond", "unipInterest"]
track_dir = "/mnt/lareaulab/reliscu/data/UCSC/hg38/genome_tracks/uniprot"

for file in glob.glob(f"{track_dir}/*.bed"):
    file_name = file.split("/")[-1].split(".bb.bed")[0]
    if file_name in keep:
        track_df = pd.read_csv(file, sep="\t")
        track_df.insert(0, "file_name", file_name)
        track_df_list.append(track_df)

/tmp/ipykernel_2312001/3072133374.py:10: DtypeWarning: Columns (0: pmids) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")
/tmp/ipykernel_2312001/3072133374.py:10: DtypeWarning: Columns (0: longName, 1: syns) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")


## Prep InterProScan results

In [5]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins_v2.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

#  "/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/data/old/interproscan/results/proteins.fa.tsv",

In [7]:
interproscan_results.head()

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways
0,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:6.10.140.620,-,283,316,1.1E-25,CATH-Gene3D,19-08-2026,-,-,-,-
1,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.30.200.20,Phosphorylase Kinase; domain 1,1,93,1.7E-39,CATH-Gene3D,19-08-2026,-,-,-,-
2,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:1.10.510.10,Transferase(Phosphotransferase) domain 1,94,282,3.5E-66,CATH-Gene3D,19-08-2026,-,-,-,-
3,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.10.450.50,-,334,486,6.9E-75,CATH-Gene3D,19-08-2026,-,-,-,-
4,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-FunFam,G3DSA:3.10.450.50:FF:000001,calcium/calmodulin-dependent protein kinase ty...,340,486,1.2E-92,CATH-FunFam,19-08-2026,-,-,-,-


In [8]:
interproscan_results.shape

(444449, 15)

In [9]:
interproscan_results.value_counts("signature_description")

signature_description
-                                              108487
Consensus disorder prediction                   54951
Transmembrane region                            13408
Coil                                            10280
Non cytoplasmic domain                           9707
                                                ...  
TARBP1 domain                                       1
SpoU rRNA Methylase family                          1
RNA methyltransferase TRMH family profile           1
N-terminal domain of synaptotagmin-1 and -2         1
Signal recognition particle 19 kDa protein          1
Name: count, Length: 24477, dtype: int64

In [12]:
# Parse the PIRSR data

with open("data/interproscan/interpro_data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [13]:
interproscan_results = interproscan_results.merge(
    pirsr_df, left_on="signature_accession", right_on="accession", how="left"
)

In [14]:
interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label']
interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_accession']
interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_accession']  

In [16]:
interproscan_results.head()

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group
0,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:6.10.140.620,-,283,316,1.1E-25,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.30.200.20,Phosphorylase Kinase; domain 1,1,93,1.7E-39,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:1.10.510.10,Transferase(Phosphotransferase) domain 1,94,282,3.5E-66,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.10.450.50,-,334,486,6.9E-75,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-FunFam,G3DSA:3.10.450.50:FF:000001,calcium/calmodulin-dependent protein kinase ty...,340,486,1.2E-92,CATH-FunFam,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Map events to domains

In [17]:
def near_exon(exon_start, exon_end, scan_start, scan_end, window=600):
    """Returns boolean mask for features within window of the exon."""
    return (
        (exon_end  >= scan_start - window) &
        (exon_start <= scan_end   + window)
    )
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def rel_cds_to_genome(cds_obj, strand):
    """
    Build a list of exon segments mapping between genomic coordinates
    and CDS-relative coordinates, in transcript (5'->3') order.

    Each segment is a dict:
        genome_start, genome_end : genomic coordinates (always genome_start <= genome_end)
        rel_start, rel_end       : CDS-relative coordinates (always rel_start <= rel_end)
    """
    cds = _cds_rows(cds_obj)

    # order exons in transcript (5'->3') order
    if strand == '-':
        cds = sorted(cds, key=lambda c: c['start'], reverse=True)
    else:
        cds = sorted(cds, key=lambda c: c['start'])

    segments = []
    cds_start = 0
    for c in cds:
        length = c['end'] - c['start'] + 1
        segments.append({
            'genome_start': c['start'],
            'genome_end': c['end'],
            'rel_start': cds_start,
            'rel_end': cds_start + length - 1,
        })
        cds_start += length

    return segments

def map_aa_to_genome(rel_to_genome, aa_start, aa_end, strand):
    result = []
    # convert AA position to relative CDS position
    # note: subtract 1 to convert from 1-indexed AA to 0-indexed
    cds_start = (aa_start - 1) * 3
    cds_end = (aa_end - 1) * 3 + 2
    for seg in rel_to_genome:
        # seg contains mapping from relative CDS position to genome position
        overlap_start = max(cds_start, seg['rel_start'])
        overlap_end = min(cds_end, seg['rel_end'])
        if overlap_start <= overlap_end:
            if strand == '-':
                # rel increases as genome decreases
                g_start = seg['genome_end'] - (overlap_end - seg['rel_start'])
                g_end = seg['genome_end'] - (overlap_start - seg['rel_start'])
            else:
                # rel increases as genome increases
                g_start = seg['genome_start'] + (overlap_start - seg['rel_start'])
                g_end = seg['genome_start'] + (overlap_end - seg['rel_start'])
            result.append((g_start, g_end))
    return result

In [18]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

#### Merge Uniprot tracks with splicing events

In [44]:
FLANK = 600

def flatten_exon_data(data: dict, flank=600) -> pd.DataFrame:
    rows = []
    for ev, rec in data.items():
        meta = rec["meta"]
        strand = rec['meta']['strand']

        def emit(type_name, d, start=None, end=None, flanked_start=None, flanked_end=None):
            row = {
                "event_id": ev,
                "chrom": meta["chrom"],
                "strand": meta["strand"],
                "gene": meta["gene"],
                "meta_es": meta["es"],
                "meta_ee": meta["ee"],
                "type": type_name,
            }
            row.update(d)  # transcript_id, exon_cds_start/end, frame_preserving, etc.
            row["start"] = start
            row["end"] = end
            row['flanked_start'] = flanked_start
            row['flanked_end'] = flanked_end 
            rows.append(row)
        
        # inclusion: use its own exon_cds_start/end
        for type_name in ("inclusion", "exon_diff_boundary_siblings", "exon_diff_junction_siblings"):
            d = rec.get(type_name)
            if d and d.get("transcript_id") is not None:
                if strand == "+":
                    flanked_start = max(d['exon_cds_start'] - flank, 0)
                    flanked_end = d['exon_cds_end'] + flank
                else:
                    flanked_start = d['exon_cds_end'] + flank 
                    flanked_end = max(d['exon_cds_start'] - flank, 0)
                    
                emit(type_name, d, 
                    start=d["exon_cds_start"], 
                    end=d["exon_cds_end"],
                    flanked_start=flanked_start,
                    flanked_end=flanked_end
                )

        # real_skip: no exon coords (it's skipped) — use flanked window around the parent exon
        d = rec.get("real_skip")
        if d and d.get("transcript_id") is not None:
            if strand == "+":
                flanked_start = meta["es"] + flank
                flanked_end = max(meta["es"] - flank, 0)
            else:
                flanked_start = meta["es"] + flank
                flanked_end = max(meta["es"] - flank, 0)
                
            emit("real_skip", d,
                start=meta["es"],
                end=meta["ee"],
                flanked_start=flanked_start,
                flanked_end=flanked_end
            )
        
    df = pd.DataFrame(rows)
    df["flanked_start"] = df["flanked_start"].astype(int)
    df["flanked_end"] = df["flanked_end"].astype(int)

    return df

long_df = flatten_exon_data(event_protein_map)

In [51]:
import bioframe as bf

cols = [
    "file_name", "chrom", "chromStart", "chromEnd", "name", "status", "annotationType", "position"
]

results = []
for track_df in track_df_list:
    # t = track_df.rename(columns={"chromStart": "start", "chromEnd": "end"})
    t = track_df.loc[:, cols]
    overlapped = bf.overlap(
        long_df, t,
        cols1=("chrom", "start", "end"),
        cols2=("chrom", "chromStart", "chromEnd"),
        suffixes=("", "_"),
        how="inner"
    )
    results.append(overlapped)

uniprot_merged = pd.concat(results, ignore_index=True)

In [ ]:
# uniprot_merged[(uniprot_merged['file_name_'] == "unipLocSignal") & (uniprot_merged['status_'].str.contains("Manually"))]

In [57]:
uniprot_by_event = {
    event_id: grp
    for event_id, grp in uniprot_merged.groupby("event_id")  # adjust column name — see note below
}

In [60]:
uniprot_by_event['ENSG00000141337_ProteinCoding_1']

,event_id,chrom,strand,gene,meta_es,meta_ee,type,transcript_id,aa_start,aa_end,exon_cds_start,exon_cds_end,frame_preserving,clean_start,clean_end,start,end,flanked_start,flanked_end,excluded_transcript_types,excluded_transcript_tags,file_name_,chrom_,chromStart_,chromEnd_,name_,status_,annotationType_,position_
20109,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,real_skip,ENST00000452479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68306943,68307711,68307543,68306343,{},{},unipOther,chr17,68307622,68307625,bind,Manually reviewed (Swiss-Prot),binding site,amino acid 44 on protein Q96EG1
20110,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,real_skip,ENST00000452479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68306943,68307711,68307543,68306343,{},{},unipOther,chr17,68307625,68307628,bind,Manually reviewed (Swiss-Prot),binding site,amino acid 45 on protein Q96EG1
20111,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,inclusion,ENST00000621439,0.0,72.0,68307494.0,68307711.0,False,True,False,68307494,68307711,68306894,68308311,NaN,NaN,unipOther,chr17,68307622,68307625,bind,Manually reviewed (Swiss-Prot),binding site,amino acid 44 on protein Q96EG1
20112,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,inclusion,ENST00000621439,0.0,72.0,68307494.0,68307711.0,False,True,False,68307494,68307711,68306894,68308311,NaN,NaN,unipOther,chr17,68307625,68307628,bind,Manually reviewed (Swiss-Prot),binding site,amino acid 45 on protein Q96EG1
21890,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,real_skip,ENST00000452479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68306943,68307711,68307543,68306343,{},{},unipLocSignal,chr17,68307493,68307541,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-16 on protein Q96EG1
21912,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,inclusion,ENST00000621439,0.0,72.0,68307494.0,68307711.0,False,True,False,68307494,68307711,68306894,68308311,NaN,NaN,unipLocSignal,chr17,68307493,68307541,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-16 on protein Q96EG1
73479,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,real_skip,ENST00000452479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68306943,68307711,68307543,68306343,{},{},unipChain,chr17,68307541,68420460,ASG,Manually reviewed (Swiss-Prot),chain,amino acids 17-525 on protein Q96EG1
73480,ENSG00000141337_ProteinCoding_1,chr17,+,ENSG00000141337,68306943,68307711,inclusion,ENST00000621439,0.0,72.0,68307494.0,68307711.0,False,True,False,68307494,68307711,68306894,68308311,NaN,NaN,unipChain,chr17,68307541,68420460,ASG,Manually reviewed (Swiss-Prot),chain,amino acids 17-525 on protein Q96EG1


In [61]:
pickle.dump(uniprot_by_event, open("data/uniprot_by_event.pkl", "wb"))

#### Merge IntroProScan results

In [62]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [77]:
event_protein_map

{'ENSG00000187634_ProteinCoding_1': {'meta': {'chrom': 'chr1',
   'strand': '+',
   'es': 931039,
   'ee': 931089,
   'gene': 'ENSG00000187634',
   'us_intron_start': 930337,
   'ds_intron_end': 935771},
  'inclusion': {'transcript_id': 'ENST00000616016',
   'aa_start': 263,
   'aa_end': 280,
   'exon_cds_start': 931039,
   'exon_cds_end': 931089,
   'frame_preserving': True,
   'clean_start': False,
   'clean_end': False},
  'real_skip': {'transcript_id': None,
   'excluded_transcript_types': {'protein_coding'},
   'excluded_transcript_tags': {'mRNA_start_NF,cds_start_NF'}},
  'exon_diff_boundary_siblings': None,
  'exon_diff_junction_siblings': None},
 'ENSG00000188290_ProteinCoding_1': {'meta': {'chrom': 'chr1',
   'strand': '-',
   'es': 999692,
   'ee': 999787,
   'gene': 'ENSG00000188290',
   'us_intron_start': 999614,
   'ds_intron_end': 999865},
  'inclusion': {'transcript_id': 'ENST00000304952',
   'aa_start': 36,
   'aa_end': 67,
   'exon_cds_start': 999692,
   'exon_cds_end'

In [ ]:
# map interproscan results to their corresponding transcript (representing the splicing event of interest)
cols = ['protein_accession', 'sequence_length', 'analysis', 
        'signature_description', 'start', 'stop', 'interpro_description']
ipr = interproscan_results.loc[:, cols]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

interproscan_by_event = defaultdict(dict)

for ev, rec in event_protein_map.items():
    
    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    
    overlap_mask = [
        near_exon(
            rec_incl['aa_start'], 
            rec_incl['aa_end'], 
            row['start'], 
            row['stop'],
            window=200
        ) 
        for _, row in incl_df.iterrows()
    ]
    overlap_df = incl_df[overlap_mask]
    
    if not overlap_df.empty:
        strand = rec['meta']['strand']

        # save AA position of protein domains in terms of genomic coordinates
        cds_obj = cds_by_transcript[rec_incl['transcript_id']]
        cds_to_genome = rel_cds_to_genome(cds_obj, strand)
        
        genome_coords = []
        for _, row in overlap_df.iterrows():
            aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
            genome_coords.append(aa_to_genome)
    
        incl_cols = ['aa_start', 'aa_end', 
                     'exon_cds_start', 'exon_cds_end',
                     'frame_preserving', 'clean_start', 'clean_end']
        
        interproscan_by_event[ev] = {
            'meta': rec['meta'],
            'inclusion': overlap_df.assign(
                **{col: rec_incl[col] for col in incl_cols},
                genome_coords=genome_coords
            ).reset_index(drop=True),
            'real_skip': None,
            'exon_diff_junction_siblings': None,
            'exon_diff_boundary_siblings': None
        }
    
        # real skip
        if rec.get('real_skip'):
            t = rec['real_skip']['transcript_id'] 
            if t is not None:
                skip_df = ipr_grouped.get(t)
                if skip_df is not None: 
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    # convert AA pos to genome position
                    genome_coords = []
                    for _, row in skip_df.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        scan_start = min(min(x) for x in aa_to_genome)
                        scan_end = max(max(x) for x in aa_to_genome)
                        # real_skip: no exon coords (it's skipped) — use window around the parent exon
                        if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                            genome_coords.append(aa_to_genome)
                        else:
                            genome_coords.append(None)
                            
                    skip_df = skip_df.assign(
                        exon_cds_start=rec['meta']['es'],
                        exon_cds_end=rec['meta']['ee'],
                        genome_coords=genome_coords
                    )
                    skip_df = skip_df[skip_df['genome_coords'].notna()]
                        
                    if not skip_df.empty:
                        interproscan_by_event[ev]['real_skip'] = skip_df.reset_index(drop=True)
                    
        # junction siblings
        if rec.get('exon_diff_junction_siblings'):
            t = rec['exon_diff_junction_siblings']['transcript_id'] 
            if t is not None:
                    sib = rec['exon_diff_junction_siblings']
                    sib_df = ipr_grouped.get(t) 
                    if sib_df is None:
                        continue
                    
                    overlap_mask = [
                        near_exon(
                            sib['aa_start'], 
                            sib['aa_end'], 
                            row['start'], 
                            row['stop'],
                            window=200
                        ) 
                        for _, row in sib_df.iterrows()
                    ]
                    
                    sib_overlap = sib_df[overlap_mask]
                    
                    if not sib_overlap.empty:
                        # save AA position of protein domains in terms of genomic coordinates
                        cds_obj = cds_by_transcript[t]
                        cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                        genome_coords = []
                        for _, row in sib_overlap.iterrows():
                            aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                            genome_coords.append(aa_to_genome)

                        entry = sib_overlap.assign(
                            aa_start=sib['aa_start'],
                            aa_end=sib['aa_end'],
                            exon_cds_start=sib['exon_cds_start'],
                            exon_cds_end=sib['exon_cds_end'],
                            genome_coords=genome_coords
                        ).reset_index(drop=True)

                        interproscan_by_event[ev]['exon_diff_junction_siblings'] = entry
                    
        # boundary siblings
        if rec.get('exon_diff_boundary_siblings'):
            t = rec['exon_diff_boundary_siblings']['transcript_id'] 
            if t is not None:
                sib = rec['exon_diff_boundary_siblings']
                sib_df = ipr_grouped.get(t) 
                if sib_df is None:
                    continue
                
                overlap_mask = [
                    near_exon(
                        sib['aa_start'], 
                        sib['aa_end'], 
                        row['start'], 
                        row['stop'],
                        window=200
                    ) 
                    for _, row in sib_df.iterrows()
                ]
                
                sib_overlap = sib_df[overlap_mask]
                
                
                if not sib_overlap.empty:
                    # save AA position of protein domains in terms of genomic coordinates
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    genome_coords = []
                    for _, row in sib_overlap.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        genome_coords.append(aa_to_genome)

                    entry = sib_overlap.assign(
                        aa_start=sib['aa_start'],
                        aa_end=sib['aa_end'],
                        exon_cds_start=sib['exon_cds_start'],
                        exon_cds_end=sib['exon_cds_end'],
                        genome_coords=genome_coords
                    ).reset_index(drop=True)

                    interproscan_by_event[ev]['exon_diff_boundary_siblings'] = entry 

In [90]:
rec.get('exon_diff_boundary_siblings')

In [89]:
rec['exon_diff_boundary_siblings']

In [91]:
rec

{'meta': {'chrom': 'chr1',
  'strand': '+',
  'es': 3730927,
  'ee': 3731065,
  'gene': 'ENSG00000078900',
  'us_intron_start': 3729449,
  'ds_intron_end': 3731462},
 'inclusion': {'transcript_id': 'ENST00000378280',
  'aa_start': 349,
  'aa_end': 395,
  'exon_cds_start': 3730927,
  'exon_cds_end': 3731065,
  'frame_preserving': False,
  'clean_start': False,
  'clean_end': True},
 'real_skip': {'transcript_id': 'ENST00000604479',
  'excluded_transcript_types': set(),
  'excluded_transcript_tags': set()},
 'exon_diff_boundary_siblings': None,
 'exon_diff_junction_siblings': {'transcript_id': 'ENST00000603362',
  'aa_start': 398,
  'aa_end': 444,
  'exon_cds_start': 3730927,
  'exon_cds_end': 3731065,
  'frame_preserving': False,
  'clean_start': False,
  'clean_end': True,
  'excluded_transcript_types': set(),
  'excluded_transcript_tags': set()}}

In [74]:
rec.get('exon_diff_boundary_siblings')

{'transcript_id': None,
 'excluded_transcript_types': {'nonsense_mediated_decay', 'protein_coding'},
 'excluded_transcript_tags': {'', 'basic,GENCODE_Primary,CCDS'}}

In [72]:
sib

'transcript_id'

In [71]:
rec['exon_diff_boundary_siblings']

{'transcript_id': None,
 'excluded_transcript_types': {'nonsense_mediated_decay', 'protein_coding'},
 'excluded_transcript_tags': {'', 'basic,GENCODE_Primary,CCDS'}}

In [ ]:
pickle.dump(interproscan_by_event, open("data/interproscan_by_event.pkl", "wb"))

### Debug

In [ ]:
# ev = "ENSG00000285043_ProteinCoding_1"
# rec = event_protein_map[ev]

In [ ]:
# # map interproscan results to their corresponding transcript (representing the splicing event of interest)
# analyses_to_exclude = ['NCBIFAM', 'SFLD']
# columns = ['protein_accession', 'sequence_length', 'analysis', 
#            'signature_description', 'start', 'stop', 'interpro_description']
# ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

# ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

# event_interproscan_map = defaultdict(dict)

# rec_incl = rec['inclusion']
# incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
# if incl_df is None:
#     continue
# overlap_df = incl_df[
#     (incl_df['start'] <= rec_incl['aa_end']) &
#     (incl_df['stop']  >= rec_incl['aa_start'])
# ]



# strand = rec['meta']['strand']

In [ ]:
# # junction siblings

# for sib in rec['exon_diff_junction_siblings']:
#     t = sib['transcript_id']
#     sib_df = ipr_grouped.get(t) 
#     if sib_df is None:
#         continue
#     sib_overlap = sib_df[
#         (sib_df['start'] <= sib['aa_end']) &
#         (sib_df['stop']  >= sib['aa_start'])
#     ]
#     break


In [ ]:
# # save AA position of protein domains in terms of genomic coordinates
# cds_obj = cds_by_transcript[t]
# cds_to_genome = rel_cds_to_genome(cds_obj, strand)
# genome_coords = []
# for idx, row in sib_overlap.iterrows():
#     aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
#     genome_coords.append(aa_to_genome)

### End debug

In [ ]:
len(interproscan_by_event)

12549

## Merge results with cell type-specific splicing events

In [ ]:
with open('data/interproscan_by_event.pkl', "wb") as file:
    pickle.dump(interproscan_by_event, file)

In [ ]:
interproscan_by_ct = dict() 
interproscan_by_ct_summary = dict()

uniprot_by_ct = dict()
uniprot_by_ct_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        ######################### UNITPROT #########################
        event_uniprot_dict = {
            ev: uniprot_by_event[ev] for ev in signif_events_df.index
            if ev in uniprot_by_event
        }
        
        interpro_result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_uniprot_dict.items()
             for bucket, df in buckets.items()
             if bucket != 'meta' and df is not None],
            ignore_index=True
        )
        
        ### TO DO ###
        
        ######################### INTERPROSCAN #########################
        event_interproscan_dict = {
            ev: interproscan_by_event[ev] for ev in signif_events_df.index 
            if ev in interproscan_by_event
        }

        interpro_result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()
             if bucket != 'meta' and df is not None],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in interpro_result.columns if c not in ('event_id', 'bucket')]
        interpro_result = interpro_result[cols]
        interproscan_by_ct[ctype] = interpro_result.merge(signif_events_df, left_on="event_id", right_index=True)
        
        # summarize interpro results per splicing event
        interproscan_by_ct_summary[ctype] = interproscan_by_ct[ctype].groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(str(v) for v in x.unique() if pd.notna(v))),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [ ]:
with open("data/interproscan_by_ct.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct, file)
    
with open("data/interproscan_by_ct_summary.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct_summary, file)

In [ ]:
with open("data/interproscan_by_ct.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct, file)
    
with open("data/interproscan_by_ct_summary.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct_summary, file)

In [ ]:
rec = interproscan_by_ct_summary['Deep_layer_glutamatergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)